# Component 2 (M2): Dual-Mission Performance Intelligence & Predictive Mission Drift

**Group ID:** `j26it326` — **Component 2 (C2)**  
**Programme:** SLIIT — 4th Year Research Project  

This research notebook provides an end-to-end evaluation of the Component 2 ML and Analytics pipeline:
1. **Exploratory Data Analysis & Benchmarking:** Real loan performance spine (7,245 loans) benchmarked against MIX Market Sri Lanka medians and World Bank Findex.
2. **Dual-Mission KPI Engine:** Financial performance (PAR30, OSS, OER, Yield) vs. Social performance (female, poverty proxy, rural outreach).
3. **Retrospective Mission Drift Index (MDI):** Mathematical index quantifying deviations from the social mission across rolling windows.
4. **Predictive Machine Learning (XGBoost):** Early warning model forecasting forward drift risk probability 1–3 months in advance based on leading operational signals.
5. **Grounded Executive Intelligence (NFR-06):** Evidence-grounded narrative generation translating multi-dimensional metrics into strategic governance recommendations.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Set paths
M2_ROOT = Path("../../m2_mission_intel").resolve()
sys.path.insert(0, str(M2_ROOT))

from kpi.kpi_engine import calculate_portfolio_kpis, calculate_social_indicators
from mdi.mdi import compute_mdi, load_settings
from features.drift_features import FEATURE_COLUMNS, compute_velocity_features
from narrative.narrative_engine import generate_grounded_narrative
from serve.batch_drift_score import score_institution_and_branches
import joblib

print("Component 2 ML Modules loaded successfully from:", M2_ROOT)

--- 
## 1. Portfolio Baseline & Prior Benchmarks

We evaluate the primary partner PAR extract (7,245 loans) anchored at July 2026. PAR30 is computed using the agreed **>4.3 weekly installments late** rule.

In [ ]:
borrowers_df = pd.read_csv(M2_ROOT / "data" / "processed" / "borrowers_m2.csv")
port_kpi = calculate_portfolio_kpis(borrowers_df)
soc_kpi = calculate_social_indicators(borrowers_df)

print("=== ANCHOR MONTH (JULY 2026) PORTFOLIO SUMMARY ===")
print(f"Total Active Loans        : {port_kpi['total_loans']:,}")
print(f"Gross Portfolio Outstanding: LKR {port_kpi['portfolio_outstanding']:,.2f}")
print(f"Average Loan Size (Ticket) : LKR {port_kpi['avg_loan_size']:,.2f}")
print(f"PAR30 Balance Ratio        : {port_kpi['par30_balance_ratio']:.2%}")
print(f"PAR30 Loan Count Ratio     : {port_kpi['par30_count_ratio']:.2%}")
print(f"Female Borrower Ratio      : {soc_kpi['female_borrower_ratio']:.2%}")
print(f"Below Poverty Line Share   : {soc_kpi['poverty_proxy_score']:.2%}")
print(f"Rural Borrower Ratio       : {soc_kpi['rural_borrower_ratio']:.2%}")

--- 
## 2. Longitudinal Trajectory & Retrospective MDI

The 12-month timeline tracks operational indicators from August 2025 to July 2026. The retrospective MDI compares the rolling 3-month window against the baseline to quantify social mission drift.

In [ ]:
mdi_df = pd.read_csv(M2_ROOT / "data" / "processed" / "mdi_monthly.csv")
print(mdi_df[["month", "score", "state", "threshold"]].to_string(index=False))

# Display July drivers
latest_trends = json.loads(mdi_df.iloc[-1]["contributingTrends"])
print(f"\nTop Drivers of MDI in {mdi_df.iloc[-1]['month']}:")
for t in latest_trends[:4]:
    print(f"  - {t['label']}: Relative Change = {t['relativeChange']:+.1%}, Contribution = +{t['contribution']:.1f} pts")

--- 
## 3. Predictive Machine Learning: XGBoost Mission Drift Model

Retrospective MDI detects drift *after* it occurs. Our predictive model forecasts the probability of crossing the MDI threshold (>=20) 1–3 months ahead based on leading indicators (loan size inflation, margin pressure, demographic shifts).

In [ ]:
metadata_path = M2_ROOT / "artifacts" / "drift_model_metadata.json"
if metadata_path.exists():
    meta = json.loads(metadata_path.read_text(encoding="utf-8"))
    print("=== TRAINED XGBOOST MODEL EVALUATION METRICS ===")
    print(f"Model Name         : {meta['model_name']}")
    print(f"Training Samples   : {meta['n_train']} train / {meta['n_test']} test")
    print(f"Classifier ROC-AUC : {meta['classifier_metrics']['roc_auc']:.4f}")
    print(f"5-Fold CV ROC-AUC  : {meta['classifier_metrics']['cv_roc_auc_mean']:.4f} (+/- {meta['classifier_metrics']['cv_roc_auc_std']:.4f})")
    print(f"Precision / Recall : {meta['classifier_metrics']['precision']:.4f} / {meta['classifier_metrics']['recall']:.4f} (F1: {meta['classifier_metrics']['f1']:.4f})")
    print(f"Regressor MAE / R2 : MAE {meta['regressor_metrics']['mae']:.2f} / R2 {meta['regressor_metrics']['r2']:.4f}")
    
    print("\nTop Leading Feature Importances:")
    for f in meta['feature_importances'][:5]:
        print(f"  {f['feature']:<25}: {f['importance']:.2%}")

--- 
## 4. Branch-Level Risk Profiling & Grounded Executive Narrative

We evaluate mission performance across all 17 individual branches and generate a strictly evidence-grounded briefing compliant with NFR-06.

In [ ]:
report = score_institution_and_branches()

print("=== GROUNDED EXECUTIVE BRIEFING (NFR-06) ===")
print(f"Severity Status    : {report['severity']}")
print(f"Headline           : {report['grounded_narrative']['headline']}")
print(f"\n[Trade-Off Analysis]\n{report['grounded_narrative']['tradeoff_analysis']}")
print("\n[Managerial Recommendations]")
for i, rec in enumerate(report['grounded_narrative']['recommendations'], 1):
    print(f"  {i}. {rec}")

print("\n=== TOP 5 HIGHEST DRIFT RISK BRANCHES ===")
b_df = pd.DataFrame(report['branch_risk_rankings'])
print(b_df[['branch', 'estimated_branch_mdi', 'female_share', 'poverty_share', 'status']].head(5).to_string(index=False))